# App analytics → Delta

Capstone requirement: **Analytics inside Delta Table about your app**.

1. Use the Coastal Ops app (sync / search / agent) so rows land in Lakebase `app_events`
2. Run this notebook on a Spark cluster
3. Inspect bronze + daily gold Delta tables

Paths (defaults):
- `/tmp/coastal_ops/app_events_bronze`
- `/tmp/coastal_ops/app_analytics_daily`
- `/tmp/coastal_ops/agent_tool_usage`

In [ ]:
%pip install -q 'databricks-sdk>=0.118.0' psycopg2-binary requests

In [ ]:
dbutils.library.restartPython()

In [ ]:
import importlib.util
from pathlib import Path

script = None
roots = [Path.cwd(), Path.cwd().parent, *list(Path.cwd().parents)[:5]]
for root in roots:
    for hit in (
        root / "notebooks" / "analytics_app_events_to_delta.py",
        root / "analytics_app_events_to_delta.py",
    ):
        if hit.exists():
            script = hit
            break
    if script is not None:
        break

if script is None:
    raise FileNotFoundError("Could not find analytics_app_events_to_delta.py")

print("Loading", script)
spec = importlib.util.spec_from_file_location("analytics_app_events_to_delta", script)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
summary = mod.main()
summary

In [ ]:
daily_path = "/tmp/coastal_ops/app_analytics_daily"
display(spark.read.format("delta").load(daily_path).orderBy("event_date", "event_type"))

In [ ]:
tools_path = "/tmp/coastal_ops/agent_tool_usage"
display(spark.read.format("delta").load(tools_path))